# Lab 4 — Three detectors, one dataset
### EP0418 · Topic 5 · 2 practical hours

**You need:** `ep0418.py` and `smart_building_sensors.csv`.

---
Today you build three anomaly detectors of rising sophistication on the same data
and compare **what each one catches and what each one misses**. The comparison is
the lab; the code is not.

## Putting the assistant's code into a TODO cell

A TODO cell has a line ending in `...`. Replace **only that line** with the
assistant's code, and keep the name on the left (`s`, `clean`, `H`). Delete any
`import` or `read_csv` lines it adds; the data is already loaded. If it used a
different name for the result, add one line at the end to rename it, e.g.
`s = hourly_df`.

---
## How every lab in this module works

You will **not** write this code from scratch. Ask your AI assistant to write it.
Your job is the part it cannot do: deciding what the right answer looks like,
reading what it gave you, and proving it did what you asked.

At the foot of each task is a **verification contract** — a few `check.*` lines.
**You may not edit them.** The AI can write any implementation it likes; it cannot
satisfy these by accident.

If a check fails, read the message. It tells you what was found and what to try
next. Failing checks are the normal state of a lab in progress, not a sign you
are behind.

## Set up Gemini for this notebook (once, 30 seconds)

Open Gemini (the spark icon at the bottom of the notebook). In its chat box, click the **sliders icon** next to **+**, choose **Configure**, paste the text below and save. Leave the mode on **Default**. Colab keeps the instructions with this notebook, so every answer follows the course rules.

```
I am new to coding. Explain any code you write in plain words: what it does to my data.
Before writing code for a bigger task, tell me your plan in plain words.
Never delete or drop rows unless I ask. If a step would remove rows, say so first.
After changing the data, show the row count before and after, and the smallest and largest values.
Codes such as -999 mean a missing reading. Point them out; do not treat them as real numbers.
Do not change any line that starts with check. Those are the lab's locked checks.
If you are not sure, say so. Do not guess.
```

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/"


def fetch(*names):
    """Download any course file that is not beside the notebook yet."""
    for name in names:
        if os.path.exists(name):
            continue
        try:
            urllib.request.urlretrieve(REPO + name, name)
            print("fetched", name)
        except Exception:
            print(f"Could not download {name}. Upload it with the folder icon on the left.")
            from google.colab import files
            files.upload()


fetch("ep0418.py")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ep0418 import check


def filled(*values):
    """True once every TODO slot above has been replaced. Otherwise says so."""
    if any(v is ... or v is None for v in values):
        print("A TODO above is still '...'. Fill it in, then run this cell again.")
        return False
    return True

CSV = "smart_building_sensors.csv"
fetch(CSV)

raw = pd.read_csv(CSV, parse_dates=["timestamp"])

# Start from clean data — reuse your Lab 3 cleaning here
df = raw.copy()
df.loc[df.humidity_pct == -999, "humidity_pct"] = np.nan
df = df.drop_duplicates(subset="timestamp", keep="first").sort_values("timestamp")
print(f"{len(df):,} rows ready")

# Placeholders so later cells give a readable message, not a NameError.
alert_fixed = alert_z = alert_ctx = z = stuck = None

## Detector 1 — Fixed threshold

The baseline. One line, and what most real buildings actually run.

In [ ]:
# TODO(you): flag every reading where co2_ppm > 1000 as `alert_fixed`
# alert_fixed = ...

if alert_fixed is None:
    print("Fill in alert_fixed above, then run this cell again.")
else:
    print(f"alerts: {alert_fixed.sum():,} of {len(df):,}  ({alert_fixed.mean():.1%})")
    print(f"fired while room EMPTY: {(alert_fixed & (df.occupancy == 0)).sum()}")
    print()
    print("alerts per week — watch what happens over time:")
    print(alert_fixed.groupby(df.timestamp.dt.isocalendar().week).sum().to_string())

### Why is it getting worse?

The alert count climbs sharply in the final weeks. **The room did not change.**

A fixed threshold cannot tell the difference between "the air got worse" and "the
sensor drifted". Yours should.

## Detector 2 — Rolling z-score

Compare each reading against its own recent past instead of a fixed number.

> Compute a rolling mean and rolling standard deviation of `co2_ppm` over a
> 24-hour window, then a z-score for each reading. Flag readings with |z| > 3.

In [ ]:
# TODO(you): build `z` and `alert_z`
z = ...
alert_z = ...

if filled(z, alert_z):
    print(f"z-score alerts: {alert_z.sum():,}")
    print(f"fixed alerts  : {alert_fixed.sum():,}")

## Detector 3 — Contextual z-score

> 28°C at 3am in an empty room is strange. 28°C at 2pm with thirty people is
> Tuesday.

Compute the z-score **within each hour-of-day bucket**, so each reading is
compared against readings from similar circumstances.

In [ ]:
# TODO(you): build `alert_ctx` — z-score computed within hour-of-day groups
alert_ctx = ...

if filled(alert_ctx):
    print(f"contextual alerts: {alert_ctx.sum():,}")

### See which alarms disappeared

Counts alone do not show *which* alerts each detector drops. Plot one week.

In [ ]:
if filled(alert_fixed, alert_z, alert_ctx):
    week = (df.timestamp >= "2026-06-15") & (df.timestamp < "2026-06-22")
    fig, ax = plt.subplots(figsize=(11, 3.5))
    ax.plot(df.timestamp[week], df.co2_ppm[week], color="0.6", lw=0.8)
    for name, mask, y in [("fixed", alert_fixed, 1280), ("rolling z", alert_z, 1240),
                          ("contextual z", alert_ctx, 1200)]:
        m = week & mask
        ax.plot(df.timestamp[m], [y] * int(m.sum()), "|", ms=8, label=f"{name} ({int(m.sum())})")
    ax.axhline(1000, ls="--", color="0.4", lw=0.7)
    ax.set_ylabel("co2_ppm"); ax.legend(loc="lower left"); plt.show()

## Task 4 — Compare them honestly

Fill this in from your own numbers:

| Detector | Alerts fired | Fired when room empty | Fooled by the drift? | Catches the stuck sensor? |
|---|---|---|---|---|
| Fixed threshold | | | | |
| Rolling z-score | | | | |
| Contextual z-score | | | | |

**None of the three catches the stuck sensor.** Work out why, then build the one
statistic that does.

The stuck sensor from Lab 3 is in **`temp_c`**, not CO2: on 14 May it reports
23.87 °C for fourteen hours. Build the detector on that channel.

> Compute the rolling standard deviation of `temp_c` over a 1-hour window
> (12 readings). Flag readings where it is exactly zero as `stuck`.

In [ ]:
# TODO(you): a stuck sensor produces no outliers — it produces no VARIATION.
#            Build a detector for it on temp_c.
stuck = ...

if filled(stuck):
    print(f"stuck-sensor readings flagged: {int(stuck.sum())}")

# ---- verification contract — do not edit ----
check.flagged_at_least(stuck, minimum=100, what="stuck readings")
check.report()

---
## Read before you run

An assistant was asked for a rolling z-score of CO2 over a 24-hour window. It wrote:

**Do not run it yet.** Read it, answer the two questions in the cell below,
and only then remove the `#` and run it to see whether you were right.

The comment says 24 hours. Each row is five minutes apart. How long is the window really?

Write down the correct argument for a 24-hour window on this data.

In [ ]:
# --- AI-generated. Read it first. ---
# r = df.co2_ppm.rolling(24)          # 24-hour window
# z = (df.co2_ppm - r.mean()) / r.std()
# alert_z = z.abs() > 3

**What is wrong with it:**

**What it would produce, and what it should produce:**

---
## Before you leave

In `verification_log.md`, answer:

- Which detector would you actually deploy, and what does it cost you?
- Which error is worse in **your** track — a false alarm, or a missed detection?

Add a **Decisions** line: the detector and threshold you chose, and why.